In [2]:
# std_methods
#
#

In [1]:
import logging, os, ipynbname, datetime, pandas
from ruamel import yaml

import great_expectations as ge
from great_expectations.core.batch import BatchRequest, RuntimeBatchRequest
from great_expectations.checkpoint import SimpleCheckpoint

In [1]:
def initialize_logging(process_name=''):
    timestamp_string = datetime.datetime.now().strftime('%Y%m%d_%H%m%S')
    timestamp_suffix = '__' + timestamp_string

    if process_name == '':
        notebook_name = ipynbname.name()
    else:
        notebook_name = process_name
    
    log_file_full_name = notebook_name+timestamp_suffix+'.log'
    log_file_full_path = '/home/hdickie/log/'+ log_file_full_name
    
    
    logger = logging.getLogger(notebook_name)
    while logger.hasHandlers():
        logging.getLogger().removeHandler(logging.getLogger().handlers[0])
    
    
    logger.setLevel(logging.DEBUG)

    # create console handler and set level to debug
    ch = logging.StreamHandler()
    ch.setLevel(logging.DEBUG)

    ch2 = logging.FileHandler(log_file_full_path, mode='w')
    ch2.setLevel(logging.DEBUG)

    # create formatter
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

    # add formatter to ch
    ch.setFormatter(formatter)
    ch2.setFormatter(formatter)

    # add ch to logger
    logger.addHandler(ch)
    logger.addHandler(ch2)
    
    return logger

In [2]:
def getValidatorAndUpdatedContext(data_source_name, data_source_path, data_asset_name, expectation_suite_name, checkpoint_name):
    context = ge.get_context()

    datasource_config = {
        "name": data_source_name,
        "class_name": "Datasource",
        "module_name": "great_expectations.datasource",
        "execution_engine": {
            "module_name": "great_expectations.execution_engine",
            "class_name": "PandasExecutionEngine",
        },
        "data_connectors": {
            "default_runtime_data_connector_name": {
                "class_name": "RuntimeDataConnector",
                "module_name": "great_expectations.datasource.data_connector",
                "batch_identifiers": ["default_identifier_name"],
            }
        },
    }

    context.test_yaml_config(yaml.dump(datasource_config))
    context.add_datasource(**datasource_config)

    batch_request = RuntimeBatchRequest(
        datasource_name=data_source_name,
        data_connector_name="default_runtime_data_connector_name",
        data_asset_name=data_asset_name,  # this can be anything that identifies this data_asset for you
        runtime_parameters={"path": data_source_path},  # Add your path here.
        batch_identifiers={"default_identifier_name": "default_identifier"},
    )

    my_checkpoint = SimpleCheckpoint(
        name="my_checkpoint",
        data_context=context,
        batch_request = batch_request
    )

    context.create_expectation_suite(
        expectation_suite_name=expectation_suite_name, overwrite_existing=True
    )

    checkpoint_config = {
        "name": checkpoint_name,
        "config_version": 1,
        "class_name": "SimpleCheckpoint",
        "validations": [
            {
                "batch_request": {
                    "datasource_name": data_source_name,
                    "data_connector_name": "default_runtime_data_connector_name",
                    "data_asset_name": data_asset_name,
                },
                "expectation_suite_name": expectation_suite_name
            }
        ],
    }
    context.add_checkpoint(**checkpoint_config)

    validator = context.get_validator(
        batch_request=batch_request, expectation_suite_name=expectation_suite_name
    )
    
    return (context, validator)

def runValidation(context,validator,checkpoint_name,df):
    validator.save_expectation_suite(discard_failed_expectations=False)
    results = context.run_checkpoint(
        checkpoint_name=checkpoint_name,
        batch_request={
            "runtime_parameters": {"batch_data": df},
            "batch_identifiers": {
                "default_identifier_name": "<YOUR MEANINGFUL IDENTIFIER>"
            },
        },
    )

    context.build_data_docs()